# Build a SciFlawBench Task

Turn one row of the annotator spreadsheet into a **SciFlawBench task line**, and append it to a `.jsonl` task file.

**How to use it**

1. Run the *Setup* cell once.
2. Fill in the *Task fields* cell.
3. Run *Build and check* - it prints the JSONL line and every problem it found.
4. Run *Append to the task file* once the line is clean.
5. Run *Re-check the task file* before opening a pull request.

The rules enforced here are the ones in [`docs/task-authoring.md`](../docs/task-authoring.md) (fields, failure modes, tools) and [`docs/failure-modes.md`](../docs/failure-modes.md) (which mode to pick).

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

# Canonical values, mirrored from the harness code. The last cell cross-checks them
# whenever the harness is importable in this kernel.
QUANTITATIVE_FAILURE_MODES = [
    "correctness",
    "correct_tool_calls",
    "code_safety",
    "robustness_against_adversarial_inputs",
]
QUALITATIVE_FAILURE_MODES = [
    "sycophancy",
    "planning",
    "reasoning",
    "uncertainty_awareness",
    "aesthetic_quality",
    "lost_context_on_multi_agent_tasks",
    "implicit_domain_knowledge",
]
FAILURE_MODES = QUANTITATIVE_FAILURE_MODES + QUALITATIVE_FAILURE_MODES

AGENTS = ["code_agent", "tool_agent"]

# src/evaluation/definitions.py
VALIDATORS = [
    "content:contains_str",
    "content:exact_str_match",
    "content:numeric_match",
    "content:numeric_within_range",
    "content:paper_json_match",
    "content:scientific_notation",
    "content:json_output",
    "format:json_output",
    "judge:rubric",
]

# src/tools/definitions.py: the six tools every default agent already carries, plus
# the opt-in ones that a task has to request through `extra_tools`.
DEFAULT_TOOLS = ["web_search", "wikipedia_search", "visit_webpage", "calculator", "current_time", "arxiv_search"]
OPT_IN_TOOLS = ["read_file", "json_answer_tool"]
ALL_TOOLS = DEFAULT_TOOLS + OPT_IN_TOOLS

TASK_ID_PLACEHOLDER = "{TASK_ID}"


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until the harness checkout is found."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "core" / "tasks.py").is_file():
            return candidate
    raise RuntimeError(f"SciFlawBench repo root not found above {start}")


def display_path(path) -> str:
    """`path` relative to the repo when possible, else as given."""
    try:
        return str(Path(path).relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
print("repo root:", REPO_ROOT)
print(f"{len(FAILURE_MODES)} failure modes, {len(VALIDATORS)} verifiers, {len(ALL_TOOLS)} tools")

repo root: /home/nicholas/Documents/SciFlawBench
11 failure modes, 9 verifiers, 8 tools


## Task Fields

Fill in every slot below. The values come straight off the submission: the **prompt** and **ground truth** from the sheet, the **failure-mode flags** you ticked, and the **tools** the item needs.

* `PROMPT` - the exact text handed to the agent. Keep it verbatim; the task id is the md5 of this text, so any edit changes the id. If the task reads a local file, write the literal `{TASK_ID}` placeholder in the path: the harness resolves it at run time, and the file must live in `data/tasks/assets/<task_id>/`.
* `QUANT_FAILURE_MODES` / `QUAL_FAILURE_MODES` - the flags that are `true` for this item, as lists. At least one is required, and 1-3 is normal.
* `EXTRA_TOOLS` - only tools the default agents do **not** have. Listing a default tool is an error: smolagents rejects duplicate tool names.
* `VALIDATOR_DEFS` - the checks that turn the ground truth into a pass/fail. **The expected answer lives here**: a task file has no separate `ground_truth` field.

In [ ]:
# EDIT ME

# File the task line is appended to (created if missing). Annotators can stage in their
# own file first, e.g. 'annotator-handle.jsonl', and merge into the official set after review.
OUTPUT_FILE = REPO_ROOT / "data" / "tasks" / "v0" / "tasks.jsonl"

# The exact prompt handed to the agent. Use triple quotes when it spans several lines.
PROMPT = ""

AGENT_ID = "code_agent"  # code_agent | tool_agent

# The failure-mode flags that are true for this item (at least one; 1-3 is normal).
QUANT_FAILURE_MODES = []  # e.g. ['correctness']
QUAL_FAILURE_MODES = []  # e.g. ['sycophancy']

# Tools the default agents do NOT have. Never list a default tool: duplicate names crash the run.
EXTRA_TOOLS = [
    # {'tool_name': 'read_file'},
    # {'tool_name': 'json_answer_tool', 'kwargs': {'required_fields': ['papers']}},
]

# The expected answer, one {'name': ..., 'kwargs': {...}} per check. See
# data/example/README.md for every verifier and its kwargs.
VALIDATOR_DEFS = [
    # {'name': 'content:numeric_match', 'kwargs': {'expected': '18.4', 'tol': 0.05}},
    # {'name': 'judge:rubric', 'kwargs': {'rubric': 'Does the trace ...? Answer true only if ...'}},
]

In [ ]:
def build_task(prompt, agent_id, quant_failure_modes, qual_failure_modes, extra_tools, validators):
    """Return one harness task dict, shaped like a task-file line."""
    prompt = str(prompt).strip()
    return {
        "task_id": hashlib.md5(prompt.encode("utf-8")).hexdigest(),
        "task": prompt,
        "agent_id": agent_id,
        "failure_modes": {
            "quantitative": list(quant_failure_modes),
            "qualitative": list(qual_failure_modes),
        },
        "extra_tools": [dict(tool) if isinstance(tool, dict) else {"tool_name": tool} for tool in extra_tools],
        "validators": [dict(validator) for validator in validators],
    }


def check_task(task) -> list[str]:
    """Return the problems found in `task` (empty when it is valid)."""
    problems = []

    prompt = str(task.get("task") or "")
    if not prompt.strip():
        problems.append("the prompt is empty")

    if task.get("agent_id") not in AGENTS:
        problems.append(f"agent_id must be one of {AGENTS}")

    modes = task.get("failure_modes") or {}
    quantitative = list(modes.get("quantitative") or [])
    qualitative = list(modes.get("qualitative") or [])
    unknown = [mode for mode in quantitative + qualitative if mode not in FAILURE_MODES]
    if unknown:
        problems.append(f"unknown failure mode(s) {unknown}; allowed: {FAILURE_MODES}")
    if not quantitative and not qualitative:
        problems.append("declare at least one failure mode (quantitative or qualitative)")

    seen_tools = []
    for index, tool in enumerate(task.get("extra_tools") or [], start=1):
        name = tool.get("tool_name") if isinstance(tool, dict) else tool
        if name not in ALL_TOOLS:
            problems.append(f"extra_tools[{index}]: unknown tool {name!r}")
        elif name in DEFAULT_TOOLS:
            problems.append(f"extra_tools[{index}]: {name} is already available, and duplicate names crash the run")
        elif name in seen_tools:
            problems.append(f"extra_tools[{index}]: {name} is listed twice")
        seen_tools.append(name)

    validators = task.get("validators") or []
    if not validators:
        problems.append("no validators: nothing checks the answer, so the task scores nothing")
    for index, validator in enumerate(validators, start=1):
        name = validator.get("name") if isinstance(validator, dict) else validator
        if name not in VALIDATORS:
            problems.append(f"validators[{index}]: unknown verifier {name!r}; allowed: {VALIDATORS}")

    if TASK_ID_PLACEHOLDER in prompt:
        assets = REPO_ROOT / "data" / "tasks" / "assets" / str(task.get("task_id"))
        if not assets.is_dir():
            problems.append(f"the prompt uses {TASK_ID_PLACEHOLDER} but {display_path(assets)} does not exist")

    return problems


task = build_task(PROMPT, AGENT_ID, QUANT_FAILURE_MODES, QUAL_FAILURE_MODES, EXTRA_TOOLS, VALIDATOR_DEFS)
problems = check_task(task)
mode_count = len(task["failure_modes"]["quantitative"]) + len(task["failure_modes"]["qualitative"])

print("task_id:", task["task_id"])
print(json.dumps(task, ensure_ascii=False))
print()
if mode_count > 3:
    print(f"note: {mode_count} failure modes. Most items probe 1-3; consider splitting the item.")
if problems:
    print("Fix these before appending:")
    for problem in problems:
        print("  -", problem)
else:
    print("Looks good. Run the next cell to append it.")

task_id: d41d8cd98f00b204e9800998ecf8427e
{"task_id": "d41d8cd98f00b204e9800998ecf8427e", "task": "", "agent_id": "code_agent", "failure_modes": {"quantitative": [], "qualitative": []}, "extra_tools": [], "validators": []}

Fix these before appending:
  - the prompt is empty
  - declare at least one failure mode (quantitative or qualitative)
  - no validators: nothing checks the answer, so the task scores nothing


In [ ]:
def read_tasks(path) -> list[dict]:
    """Every task already in `path` (an empty list when the file does not exist)."""
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def append_task(task, path) -> int:
    """Append `task` as one JSON line, refusing a duplicate prompt. Returns the new count."""
    path = Path(path)
    rows = read_tasks(path)
    if any(row.get("task_id") == task["task_id"] for row in rows):
        raise ValueError(f"a task with the same prompt is already in {display_path(path)}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(task, ensure_ascii=False) + "\n")
    return len(rows) + 1


if problems:
    print("Nothing was appended. Fix the problems above, then run this cell again.")
else:
    total = append_task(task, OUTPUT_FILE)
    print(f"appended task #{total} to {display_path(OUTPUT_FILE)}")

Nothing was appended. Fix the problems above, then run this cell again.


In [ ]:
# Re-check every task in OUTPUT_FILE with the same rules. Run this before a pull request.
rows = read_tasks(OUTPUT_FILE)
ids = [row.get("task_id") for row in rows]
duplicates = sorted({value for value in ids if ids.count(value) > 1})

problems = []
for index, row in enumerate(rows, start=1):
    for problem in check_task(row):
        problems.append(f"line {index}: {problem}")
    expected = hashlib.md5(str(row.get("task") or "").encode("utf-8")).hexdigest()
    if row.get("task_id") != expected:
        problems.append(f"line {index}: task_id does not match md5(task)")

print(f"{display_path(OUTPUT_FILE)}: {len(rows)} task(s)")
print("duplicate ids:", duplicates or "none")
for problem in problems:
    print("  -", problem)
print("All tasks are valid." if not problems and not duplicates else "Problems found.")

data/tasks/v0/tasks.jsonl: 0 task(s)
duplicate ids: none
All tasks are valid.
